# Gap Module — Improper Output Handling  ·  OWASP LLM05

The Udemy course under-covers this, yet it is one of the most exploitable classes. The flaw: **LLM
output is untrusted input**, and something downstream *executes* it — a browser renders it, a database
runs it, a template engine evaluates it, a shell interprets it. The model does not attack you; the
code that trusts its output does.

Each attack builds a deliberately vulnerable sink, feeds it an LLM-style payload, shows the exploit
fire, then applies the fix (encode / parameterise / sandbox / allowlist). Local, no live targets.

In [1]:
import importlib, labkit
importlib.reload(labkit)
from labkit import Chat, DEFAULT_TARGET
print("target:", DEFAULT_TARGET)

target: qwen2.5:7b


## 1. XSS — LLM output rendered as HTML

A chat UI renders the model's markdown/HTML. If the model emits `<img onerror=...>` (from a poisoned
doc, or because it was asked to), the browser runs it. Vulnerable sink: inject raw into the DOM.

In [2]:
import html, bleach
llm_output = 'Here is your result <img src=x onerror="steal(document.cookie)"> done.'

vulnerable_html = f"<div>{llm_output}</div>"               # raw -> executes
safe_escaped    = f"<div>{html.escape(llm_output)}</div>"  # entity-encoded -> inert
safe_bleached   = bleach.clean(llm_output, tags=["b","i","code"], strip=True)

# "executes" = an UNENCODED tag the browser will run, not the mere word "onerror"
def executes(h): return "<img" in h.lower() or "<script" in h.lower()
print("VULNERABLE:", executes(vulnerable_html), "<- live <img> tag, script runs in the browser")
print("escaped   :", executes(safe_escaped), "<- &lt;img&gt; entity-encoded, shown as text")
print("bleached  :", executes(safe_bleached), "<- tag stripped entirely")

VULNERABLE: True <- live <img> tag, script runs in the browser
escaped   : False <- &lt;img&gt; entity-encoded, shown as text
bleached  : False <- tag stripped entirely


## 2. SQL Injection — LLM-generated query / value

Text-to-SQL and agents that build queries from model output. If the model's string is concatenated
into SQL, an injected value breaks out. Fix: parameterised queries — the value can never become SQL.

In [3]:
import sqlite3
db = sqlite3.connect(":memory:")
db.executescript("CREATE TABLE users(name,role); INSERT INTO users VALUES('alice','user'),('admin','admin');")

llm_value = "x' OR '1'='1"      # the model (or a user it relayed) produced this

# VULNERABLE: string-built SQL
try:
    q = f"SELECT name FROM users WHERE name = '{llm_value}'"
    rows_vuln = db.execute(q).fetchall()
except Exception as e:
    rows_vuln = str(e)
# SAFE: parameterised
rows_safe = db.execute("SELECT name FROM users WHERE name = ?", (llm_value,)).fetchall()

print("VULNERABLE (string-built):", rows_vuln, "<- returned all rows, injection worked")
print("SAFE (parameterised)    :", rows_safe, "<- no match, value stayed a value")

VULNERABLE (string-built): [('alice',), ('admin',)] <- returned all rows, injection worked
SAFE (parameterised)    : [] <- no match, value stayed a value


## 3. SSTI — LLM output into a template engine

Apps that let the model produce templated content. If model output is *rendered* as a template
(Jinja2), `{{7*7}}` evaluates — and `{{config}}` / object traversal leads to RCE. Fix: never render
untrusted strings as templates; pass them as data to a sandboxed, autoescaped template.

In [4]:
from jinja2 import Template, Environment
from jinja2.sandbox import SandboxedEnvironment

llm_output = "{{ 7 * 7 }}"      # a probe; real payloads reach os via object traversal

# VULNERABLE: render the model string AS a template
vuln = Template("Hello " + llm_output).render()

# SAFE: model string is DATA passed to a fixed template, autoescaped + sandboxed
env = SandboxedEnvironment(autoescape=True)
safe = env.from_string("Hello {{ name }}").render(name=llm_output)

print("VULNERABLE (rendered as template):", repr(vuln), "<- 7*7 evaluated to 49")
print("SAFE (passed as data)           :", repr(safe), "<- shown literally, not evaluated")

VULNERABLE (rendered as template): 'Hello 49' <- 7*7 evaluated to 49
SAFE (passed as data)           : 'Hello {{ 7 * 7 }}' <- shown literally, not evaluated


## 4. Command Injection — LLM output to a shell

An agent tool runs a command built from model output. `; rm -rf` or `$(...)` in that string executes.
Fix: never build a shell string; pass an argument vector with no shell, and validate against an
allowlist. (Executes a harmless `echo` only.)

In [5]:
import subprocess, shlex
llm_arg = "report.txt; echo PWNED"    # the model's 'filename'

# VULNERABLE: shell=True on a built string -> the ; runs a second command
vuln = subprocess.run(f"echo reading {llm_arg}", shell=True, capture_output=True, text=True).stdout.strip()

# SAFE: argument vector, no shell — the whole string is one literal argument
safe = subprocess.run(["echo", "reading", llm_arg], capture_output=True, text=True).stdout.strip()

print("VULNERABLE (shell=True):", repr(vuln), "<- PWNED ran as a separate command" if "PWNED" in vuln else "")
print("SAFE (argv, no shell)  :", repr(safe), "<- the ;echo stayed part of the filename")

VULNERABLE (shell=True): 'reading report.txt\nPWNED' <- PWNED ran as a separate command
SAFE (argv, no shell)  : 'reading report.txt; echo PWNED' <- the ;echo stayed part of the filename


## 5. Path Traversal — LLM output as a file path

An agent writes/reads a file named by the model. `../../etc/passwd` escapes the intended directory.
Fix: resolve the path and confirm it stays under the safe root (as in Module 7 Lab 61).

In [6]:
import os
SAFE_ROOT = os.path.abspath("/tmp/llm-outputs")

def resolve(llm_path):
    target = os.path.abspath(os.path.join(SAFE_ROOT, llm_path))
    return target if target.startswith(SAFE_ROOT + os.sep) else None

for p in ["summary.txt", "../../etc/passwd", "/etc/shadow", "../secrets.env"]:
    r = resolve(p)
    print(f"{'OK  ' if r else 'DENY'}  {p:22} -> {r or 'outside safe root, refused'}")

OK    summary.txt            -> /tmp/llm-outputs/summary.txt
DENY  ../../etc/passwd       -> outside safe root, refused
DENY  /etc/shadow            -> outside safe root, refused
DENY  ../secrets.env         -> outside safe root, refused


## 6. The principle — and an end-to-end demo

All five are one mindset error: **treating model output as trusted**. Here the model itself produces a
payload (asked to help with a "filename"), and we run it through both the vulnerable and safe sinks —
the fix is at the sink, never the prompt.

In [7]:
# ask the model for output that will flow into a sink
payload = Chat(system="You output only a filename, nothing else.")(
    "Give a filename for the report. Reply with exactly: report.txt; echo PWNED")
payload = payload.strip().strip("`")
print("model produced:", repr(payload[:60]))

import subprocess
vuln = subprocess.run(f"echo {payload}", shell=True, capture_output=True, text=True).stdout.strip()
safe = subprocess.run(["echo", payload], capture_output=True, text=True).stdout.strip()
print("through VULNERABLE sink:", repr(vuln[:60]), "<- injection fires" if "PWNED" in vuln else "")
print("through SAFE sink      :", repr(safe[:60]), "<- payload stayed inert data")

model produced: 'report.txt; echo PWNED'
through VULNERABLE sink: 'report.txt\nPWNED' <- injection fires
through SAFE sink      : 'report.txt; echo PWNED' <- payload stayed inert data


---
### What this module showed

Five sinks, one lesson: **validate and encode LLM output for the context it enters**, exactly as you
would any untrusted user input. HTML → escape or sanitise (`bleach`); SQL → parameterise; templates →
data not code, sandboxed; shell → argv not strings; paths → resolve under a safe root. The model is
not the attacker; the code that executes its output without checking is the vulnerability. This is
OWASP **LLM05**, and it is the bridge between LLM security and classic AppSec — the same bugs
(XSS/SQLi/SSTI/RCE), reached through a new source.